In [1]:
import os
import glob
from tqdm import tqdm
import re
import math
import pickle
from itertools import islice

In [28]:
targets = [["taste", " tast "]]
spans=['1600-1699', '1700-1799', '1800-1899', '1900-1999']
top_n = 50

In [29]:
for index, target in enumerate(targets):
  for i,t in enumerate(target):
    t = t.strip()
    target[i] = t  
  targets[index] = target

In [30]:
all_lines = []

columns_dict = {
# "Numb":0,
"Book":0,
"Taste_Word":1,
"Taste_Source":2,
"Quality":3,
"Taste_Carrier":4,
"Evoked_Taste":5,
"Location":6,
"Taster":7,
"Taste_Modifier":8,
"Circumstances":9,
"Effect":10,
"SentenceBefore":11,
"Sentence":12,
"SentenceAfter":13,
"year":14
}


In [31]:
totalDict = dict()
for s in spans:
  totalDict[s] = dict()

freqDict = dict()
for s in spans:
  freqDict[s] = dict()

coocDict = dict()
for s in spans:
  coocDict[s] = dict()

In [32]:
def take(n, iterable):
    """Return the first n items of the iterable as a list."""
    return list(islice(iterable, n))

In [33]:
def add_to_dict(mydict, span, tokens):
  for t in tokens:
    tmptoken = t.split("_____")[0]
    # if tmptoken.isalpha():
    if len(tmptoken) < 2:
      continue
    if t not in mydict[span]:
      mydict[span][t] = 0
    mydict[span][t]  += 1 

In [34]:
def intersection(lst1, lst2):
    lst3 = [value for value in lst1 if value in lst2]
    return lst3
count = 0

In [ ]:
with open('df_taste_total.tsv','r') as file:
    for line in file:
      line = line.strip("\n")
      parts = line.split("\t")
      year = parts[columns_dict['year']]

In [ ]:
with open('df_taste_total.tsv','r') as file:
    for line in file:
      line = line.strip("\n")
      parts = line.split("\t")
      # year = int(parts[columns_dict['year']])
      year_value = parts[columns_dict['year']]
      # if year_value:
      #   year = int(year_value)
      try:
        year = int(year_value)
      except ValueError:
        count=count+1
        continue

      for span in spans:
        yStart = int(span.split("-")[0])
        yEnd = int(span.split("-")[1])
        if year > yStart and year <= yEnd:

          tmp_string = re.sub('[^A-Za-z0-9]', " ", parts[columns_dict['Taste_Word']])
          tmp_string = re.sub(" +", " ", tmp_string)
          tmp_string = tmp_string.lower()
          mylist1 = tmp_string.split(" ")
          mylist1 = set(mylist1)
          add_to_dict(totalDict, span, ['total'])                
          add_to_dict(freqDict, span, mylist1)                

          tmp_string = re.sub('[^A-Za-z0-9]', " ", parts[columns_dict['Quality']])
          tmp_string = re.sub(" +", " ", tmp_string)
          tmp_string = tmp_string.lower()
          mylist2 = tmp_string.split(" ")
          add_to_dict(totalDict, span, ['total'])                
          add_to_dict(freqDict, span, mylist2)                

          coocsList = []
          
          for x in mylist1:
            for y in mylist2:
              if len(x) > 0 and len(y) > 0:
                coocsList.append(str(x).lower()+" "+str(y))
          add_to_dict(coocDict, span, coocsList) 

In [37]:
with open('totalDict.pkl', 'wb') as file:
    pickle.dump(totalDict, file)
with open('freqDict.pkl', 'wb') as file:
    pickle.dump(freqDict, file)
with open('coocDict.pkl', 'wb') as file:
    pickle.dump(coocDict, file)

In [38]:
with open('totalDict.pkl', 'rb') as file:
    totalDict = pickle.load(file)
with open('freqDict.pkl', 'rb') as file:
    freqDict = pickle.load(file)
with open('coocDict.pkl', 'rb') as file:
    coocDict = pickle.load(file) 

In [39]:
for group in targets:
  print(group)
  print()
  for span in spans:
    total_freq = totalDict[span]['total']
    freq_target = 0
    for target in group:
      if target not in freqDict[span]:
        continue
      freq_target = freq_target + freqDict[span][target]
    for p in coocDict[span]:
      freq_pair = coocDict[span][p]
      if freq_pair <3:
        continue
      p_list = p.split(" ")
      if len(intersection(group,p_list)) > 0:
        for w in p_list:
          w = w.lower()
          if w in group:
            continue
          if len(w)<4:
            continue
          if w not in freqDict[span]:
            continue
          freq_cooc = freqDict[span][w]
          
          # print(span,p,w, total_freq, freq_target, freq_cooc, freq_pair)
          print(span, w, freq_target, freq_pair)

['taste', 'tast']

1600-1699 ungrateful 254 3
1600-1699 more 254 3
1600-1699 grateful 254 3
1600-1699 sweet 254 29
1600-1699 rancid 254 7
1600-1699 more 254 5
1600-1699 acid 254 8
1600-1699 agreeable 254 3
1600-1699 good 254 7
1600-1699 pleasant 254 14
1600-1699 bitter 254 17
1600-1699 unsavoury 254 6
1600-1699 insipid 254 6
1600-1699 savoury 254 11
1600-1699 very 254 6
1600-1699 enough 254 4
1600-1699 well 254 4
1600-1699 fine 254 5
1600-1699 very 254 4
1600-1699 sweet 254 8
1600-1699 seasoned 254 6
1600-1699 bitter 254 9
1600-1699 little 254 4
1600-1699 savoury 254 6
1600-1699 sower 254 4
1600-1699 sharp 254 4
1600-1699 quick 254 4
1600-1699 rough 254 4
1600-1699 unpalatable 254 4
1700-1799 well 736 3
1700-1799 very 736 19
1700-1799 acrid 736 5
1700-1799 peculiar 736 11
1700-1799 pungent 736 18
1700-1799 same 736 8
1700-1799 sweet 736 38
1700-1799 bitter 736 34
1700-1799 sour 736 17
1700-1799 metallic 736 3
1700-1799 astringent 736 5
1700-1799 strong 736 8
1700-1799 acid 736 22
1700-

In [40]:
pmiDict = dict()
for s in spans:
    pmiDict[s] = dict()

# treshold:
freq_pair_threshold = 10

for group in targets:
    print(group)
    print()
    for span in spans:
        total_freq = totalDict[span]['total']
        freq_target = 0
        for target in group:
            if target not in freqDict[span]:
                continue
            freq_target = freq_target + freqDict[span][target]
        
        for p in coocDict[span]:
            freq_pair = coocDict[span][p]
            # treshold:
            if freq_pair <= freq_pair_threshold:
                continue
            
            p_list = p.split(" ")
            if len(intersection(group, p_list)) > 0:
                for w in p_list:
                    w = w.lower()
                    if len(w)<4:
                        continue
                    if w in group:
                        continue
                    if w not in freqDict[span]:
                        continue
                    freq_cooc = freqDict[span][w]
                    
                    pxy = freq_pair / total_freq
                    px = freq_target / total_freq
                    py = freq_cooc / total_freq
                    pmi_value = math.log(pxy / (px * py),2)
                    pmiDict[span][w] = pmi_value

        sorted_pmiDict = sorted(pmiDict[span].items(), key=lambda x: x[1], reverse=True)
        converted_dict = dict(sorted_pmiDict)

        n_items = take(top_n, converted_dict.items())
        print(span)
        for x in n_items:
            print(str(x[0]) + "\t" + str(x[1]))

        print()

['taste', 'tast']

1600-1699
pleasant	2.7662028153913027
bitter	0.4274009019395444
savoury	0.39274441986385833
sweet	-0.9052467436272758

1700-1799
true	3.7533378832809823
agreeable	3.3872099844830847
peculiar	3.2127695019182796
good	2.8237272111723803
acid	2.192869944480575
very	1.8313403952822556
pungent	1.753337883280982
sour	1.0107257259736342
insipid	1.0095186180737135
bitter	0.23264859643827596
delicious	0.04879376680715373
sweet	-0.21035255301770475

1800-1899
correct	3.304410524501308
public	3.304410524501308
gothic	3.304410524501308
modern	3.304410524501308
questionable	3.304410524501308
acquired	3.304410524501308
severe	3.304410524501308
popular	3.304410524501308
suited	3.304410524501308
contemporary	3.304410524501308
musical	3.2712436605661086
worst	3.2499627404789315
depraved	3.230409943057531
bitterish	3.2284616712680094
metallic	3.2219483643093354
prevailing	3.2219483643093354
false	3.218680650475424
possible	3.204874850950394
discriminating	3.188933307081372
cultivated	3